<a href="https://colab.research.google.com/github/betulbilhan2/xai-hate-speech-detection/blob/main/HateXplain_Binary_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HateXplain Binary Classification Pipeline
**3-sınıf (hate/offensive/normal) → 2-sınıf (toxic/normal) tam pipeline.**

Colab'de her hücreyi sırayla çalıştırın.

## HÜCRE 1: KURULUM

In [ ]:
!pip install -q -U transformers datasets evaluate lime shap accelerate

from google.colab import drive
drive.mount('/content/drive')

import requests, json, re, io
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict, concatenate_datasets

## HÜCRE 2: VERİ ÇEKME

In [ ]:
print("📦 Ham veriler çekiliyor...")
data_raw = requests.get("https://raw.githubusercontent.com/punyajoy/HateXplain/master/Data/dataset.json").json()
divisions = requests.get("https://raw.githubusercontent.com/punyajoy/HateXplain/master/Data/post_id_divisions.json").json()

def create_split(split_name):
    split_data = []
    for post_id in divisions[split_name]:
        item = data_raw[post_id]
        split_data.append({
            "id": post_id,
            "post_tokens": item["post_tokens"],
            "text": " ".join(item["post_tokens"]),
            "annotators": item["annotators"],
            "rationales": item["rationales"]
        })
    return Dataset.from_list(split_data)

train_ds = create_split('train')
val_ds   = create_split('val')
test_ds  = create_split('test')
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## HÜCRE 3: ÖNCEKİ 3-SINIF DAĞILIM

In [ ]:
def get_majority(annotators):
    labels = [a["label"] for a in annotators]
    return max(set(labels), key=labels.count)

labels_3 = [get_majority(x["annotators"]) for x in train_ds]
df3 = pd.DataFrame(labels_3, columns=['label'])
print("--- 3-Sınıf Dağılımı (Train) ---")
print(df3['label'].value_counts())

## HÜCRE 4: TEMİZLİK (Anlaşmazlık + Rationale + Metin)

In [ ]:
def check_disagreement(x):
    return len(set([a["label"] for a in x["annotators"]])) == 3

def has_rationales(x):
    majority = get_majority(x["annotators"])
    if majority in ["hatespeech", "offensive"]:
        if not x["rationales"] or sum([sum(r) for r in x["rationales"]]) == 0:
            return False
    return True

def clean_text(x):
    t = x["text"]
    t = t.replace('<user>', '')
    t = re.sub(r"http\S+|www\S+|https\S+", '', t, flags=re.MULTILINE)
    t = re.sub(r"@\w+", "", t)
    t = re.sub(r"#(\w+)", r"\1", t)
    t = re.sub(r'\s+', ' ', t).strip()
    x["text"] = t
    return x

train_clean = train_ds.filter(lambda x: not check_disagreement(x))
train_clean = train_clean.filter(has_rationales)
train_clean = train_clean.map(clean_text)
val_ds  = val_ds.map(clean_text)
test_ds = test_ds.map(clean_text)
print(f"Temizlik sonrası train: {len(train_clean)}")

## HÜCRE 5: BINARY ETİKET BİRLEŞTİRME

In [ ]:
print("""
🔄 ETİKET BİRLEŞTİRME:
  hatespeech + offensive → 0 (Toxic)
  normal                 → 1 (Normal)
""")

def create_binary_gt(example):
    labels = [a["label"] for a in example["annotators"]]
    majority = max(set(labels), key=labels.count)
    example["final_label"] = 0 if majority in ["hatespeech", "offensive"] else 1

    token_len = len(example["post_tokens"])
    if majority in ["hatespeech", "offensive"] and example["rationales"]:
        arrs = []
        for r in example["rationales"]:
            rl = list(r)
            if len(rl) < token_len:   rl.extend([0]*(token_len - len(rl)))
            elif len(rl) > token_len: rl = rl[:token_len]
            arrs.append(np.array(rl))
        if arrs:
            example["final_rationale"] = (np.sum(arrs, axis=0) >= 2).astype(int).tolist()
        else:
            example["final_rationale"] = [0]*token_len
    else:
        example["final_rationale"] = [0]*token_len
    return example

train_ready = train_clean.map(create_binary_gt)
val_ready   = val_ds.map(create_binary_gt)
test_ready  = test_ds.map(create_binary_gt)

# Dağılım kontrolü
bl = [x["final_label"] for x in train_ready]
print(f"Toxic: {bl.count(0)}, Normal: {bl.count(1)}, Oran: {bl.count(0)/max(bl.count(1),1):.2f}")

## HÜCRE 6: TOKENİZASYON

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_align(example):
    tok = tokenizer(example["post_tokens"], is_split_into_words=True,
                    truncation=True, padding="max_length", max_length=128)
    wids = tok.word_ids()
    aligned = []
    for wi in wids:
        if wi is None: aligned.append(0)
        elif wi < len(example["final_rationale"]): aligned.append(example["final_rationale"][wi])
        else: aligned.append(0)
    tok["aligned_rationale"] = aligned
    tok["labels"] = example["final_label"]
    return tok

train_tok = train_ready.map(tokenize_align)
val_tok   = val_ready.map(tokenize_align)
test_tok  = test_ready.map(tokenize_align)

cols = ["input_ids", "attention_mask", "labels", "aligned_rationale"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

## HÜCRE 7: SENTETİK VERİ

In [ ]:
synthetic_csv = """text;old_label
I swear to god this steak is so fucking good I could cry;0
My boy absolutely killed that presentation, deadass snatched their wigs;0
That movie was pure dogshit but we laughed our asses off the entire time;0
Motherfuckers really thought we were gonna lose this tournament, yeah right;0
You are a crazy bitch for pulling off that surprise party, love you;0
This beat goes stupid hard, the producer is a fucking maniac;0
Look at this sick ass car, I would literally kill to drive it;0
Bro is built different, absolutely shitting on the competition today;0
We are getting absolutely trashed tonight, who is bringing the tequila;0
This new update is a massive pain in my dick but I will figure it out;0
Shawty looking fine as hell in that dress, no cap;0
He is a stupid genius for coming up with that workaround;0
I am so fucking tired of this rain, I just want to see the sun;0
That test kicked my ass so hard I am seeing stars;0
You goofy bastard, I missed you so much;0
Man this pizza is the bomb dot com, absolutely slapping right now;0
They really tried to play us but we secured the fucking bag;0
You are literally the most useless sack of garbage I have ever met;1
Shut your ugly mouth before I make you swallow your own teeth;1
Anyone who drives that slow in the fast lane deserves to crash;1
Your brain is smoother than a bowling ball, absolutely zero critical thinking skills;1
I hope your dog runs away from you because you are a terrible owner;1
Why do you always dress like a blind clown in a dumpster;1
Stop talking, every word that comes out of your mouth makes everyone in the room dumber;1
You absolute failure, you could not organize a bake sale in a bakery;1
Get out of my lobby you trash tier scrub, uninstall the game;1
I would rather drag my face across broken glass than listen to your podcast;1
You are a complete joke and nobody here respects you;1
Your cooking tastes like actual vomit, never invite me over again;1
Look at this sweaty tryhard thinking he is actually good at life;1
You must have been dropped on your head repeatedly as a child to be this dense;1
I am blocking you because your personality is toxic radiation;1
Kick rocks you ungrateful little brat;1
You are a parasitic leech who only takes from your friends;1
We must secure the existence of our people and a future for our children;2
It is just basic biology that men are naturally suited for leadership while women are better at nurturing;2
I am not saying they are all criminals, but look at the statistics of who commits the most violence in our cities;2
Their culture is simply incompatible with western values of freedom and democracy;2
Marriage has always been defined one way for millennia, why should we change the natural order now;2
You are so well spoken and articulate for someone from your neighborhood;2
Globalists with dual loyalties are secretly controlling the media and banks to undermine our sovereignty;2
If they just complied with the officers and stopped resisting, these unfortunate incidents would not happen;2
It is a biological fact that certain populations evolved with higher average IQs than others;2
We need to protect our neighborhoods from this sudden influx of urban diversity;2
I have nothing against them personally but I just do not want their lifestyle pushed onto my kids at school;2
They are taking all the resources that should belong to native born citizens first;2
People with those kinds of genetic defects should be strongly discouraged from reproducing;2
True patriots understand that our demographic replacement is being orchestrated on purpose;2
Let us be honest, they only got into that prestigious university because of quotas, not merit;2
Society was much more stable before we allowed alternative lifestyles to become mainstream;2"""

df_syn = pd.read_csv(io.StringIO(synthetic_csv), sep=";")
# Eski: 0=Normal,1=Offensive,2=Hate → Yeni: 0=Toxic,1=Normal
df_syn["labels"] = df_syn["old_label"].map({0: 1, 1: 0, 2: 0})

syn_hf = Dataset.from_pandas(df_syn[['text', 'labels']])
syn_tok = syn_hf.map(lambda x: tokenizer(x['text'], padding="max_length", truncation=True, max_length=128), batched=True)
syn_tok.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

combined_train = concatenate_datasets([train_tok, syn_tok]).shuffle(seed=42)
print(f"Toplam eğitim verisi: {len(combined_train)}")

## HÜCRE 8: METRİKLER

In [ ]:
import evaluate
from sklearn.metrics import accuracy_score, f1_score, classification_report

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    return {
        'accuracy': accuracy_score(eval_pred.label_ids, preds),
        'f1': f1_score(eval_pred.label_ids, preds, average='macro')
    }

## HÜCRE 9: USTA AŞÇI (BERT) EĞİTİMİ — 2 SINIF

In [ ]:
from transformers import BertForSequenceClassification, TrainingArguments, Trainer

print("🚀 Usta Aşçı (BERT) — 2 Sınıf (Toxic/Normal) eğitimi başlıyor...")
bert_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

bert_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/HateSpeech/bert-binary-teacher",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01,
    load_best_model_at_end=True, logging_steps=100,
)

bert_trainer = Trainer(
    model=bert_model, args=bert_args,
    train_dataset=combined_train, eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)
bert_trainer.train()

## HÜCRE 10: BERT TEST SONUÇLARI

In [ ]:
bert_preds = bert_trainer.predict(test_tok)
bert_pred_labels = np.argmax(bert_preds.predictions, axis=1)
bert_true = [x["labels"].item() for x in test_tok]

print("\n📋 BERT (Usta) — Binary Test Sonuçları:")
print(classification_report(bert_true, bert_pred_labels, target_names=["Toxic","Normal"], digits=4))

## HÜCRE 11: ÇIRAK AŞÇI (DistilBERT) EĞİTİMİ — 2 SINIF

In [ ]:
from transformers import DistilBertForSequenceClassification

print("🚀 Çırak Aşçı (DistilBERT) — 2 Sınıf eğitimi başlıyor...")
student_model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

student_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/HateSpeech/distilbert-binary-student",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01,
    load_best_model_at_end=True, logging_steps=100,
)

student_trainer = Trainer(
    model=student_model, args=student_args,
    train_dataset=combined_train, eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)
student_trainer.train()

## HÜCRE 12: DistilBERT TEST SONUÇLARI

In [ ]:
st_preds = student_trainer.predict(test_tok)
st_pred_labels = np.argmax(st_preds.predictions, axis=1)

print("\n📋 DistilBERT (Çırak) — Binary Test Sonuçları:")
print(classification_report(bert_true, st_pred_labels, target_names=["Toxic","Normal"], digits=4))

## HÜCRE 13: LIME ANALİZİ — USTA (BERT)

In [ ]:
from lime.lime_text import LimeTextExplainer
from transformers import BertTokenizerFast

bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
class_names_binary = ["Toxic", "Normal"]
lime_explainer = LimeTextExplainer(class_names=class_names_binary)

def predict_proba_bert(texts):
    enc = bert_tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    enc = {k: v.to(bert_model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = bert_model(**enc)
        probs = torch.nn.functional.softmax(out.logits, dim=-1)
    return probs.cpu().numpy()

# Formatı resetle (metin ve rationale erişimi için)
test_tok.reset_format()

sample_idx = 42
sample_text = test_tok[sample_idx]['text']
true_label = test_tok[sample_idx]['labels']
print(f"--- İNCELENEN ÖRNEK ---")
print(f"Metin: {sample_text}")
print(f"Gerçek Sınıf: {class_names_binary[true_label]}")

exp = lime_explainer.explain_instance(sample_text, predict_proba_bert, num_features=10)
exp.show_in_notebook(text=True)

## HÜCRE 14: LIME PLAUSIBILITY — USTA (BERT)

In [ ]:
from sklearn.metrics import f1_score as f1_metric

num_samples = len(test_tok)
np.random.seed(42)
sample_indices = np.arange(num_samples)

bert_plaus_scores = []
print(f"🚀 BERT için {num_samples} örnek üzerinden Plausibility hesaplanıyor...")

for idx in sample_indices.tolist():
    text = test_tok[idx]['text']
    hr = test_tok[idx]['rationales']
    hr_arr = np.array(hr)
    if hr_arr.size == 0 or np.sum(hr_arr) == 0: continue
    hr_1d = np.max(hr_arr, axis=0).tolist() if hr_arr.ndim > 1 else hr_arr.tolist()

    try:
        e = lime_explainer.explain_instance(text, predict_proba_bert, num_features=5)
        feats = e.as_list()
    except: continue

    imp_words = [f[0].lower() for f in feats if f[1] > 0]
    words = text.split()
    ml = min(len(words), len(hr_1d))
    yt = hr_1d[:ml]
    yp = [1 if words[i].lower() in imp_words else 0 for i in range(ml)]
    bert_plaus_scores.append(f1_metric(yt, yp, zero_division=0))

if bert_plaus_scores:
    print(f"\n🎯 BERT Plausibility: %{np.mean(bert_plaus_scores)*100:.2f} ({len(bert_plaus_scores)} örnek)")

## HÜCRE 15: LIME PLAUSIBILITY — ÇIRAK (DistilBERT)

In [ ]:
from transformers import DistilBertTokenizerFast
student_tokenizer_fast = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def predict_proba_student(texts):
    enc = student_tokenizer_fast(texts, padding=True, truncation=True, return_tensors="pt")
    enc = {k: v.to(student_model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = student_model(**enc)
        probs = torch.nn.functional.softmax(out.logits, dim=-1)
    return probs.cpu().numpy()

student_plaus_scores = []
print(f"🚀 DistilBERT için {num_samples} örnek üzerinden Plausibility hesaplanıyor...")

for idx in sample_indices.tolist():
    text = test_tok[idx]['text']
    hr = test_tok[idx]['rationales']
    hr_arr = np.array(hr)
    if hr_arr.size == 0 or np.sum(hr_arr) == 0: continue
    hr_1d = np.max(hr_arr, axis=0).tolist() if hr_arr.ndim > 1 else hr_arr.tolist()

    try:
        e = lime_explainer.explain_instance(text, predict_proba_student, num_features=5)
        feats = e.as_list()
    except: continue

    imp_words = [f[0].lower() for f in feats if f[1] > 0]
    words = text.split()
    ml = min(len(words), len(hr_1d))
    yt = hr_1d[:ml]
    yp = [1 if words[i].lower() in imp_words else 0 for i in range(ml)]
    student_plaus_scores.append(f1_metric(yt, yp, zero_division=0))

if student_plaus_scores:
    bp = np.mean(bert_plaus_scores)*100 if bert_plaus_scores else 0
    sp = np.mean(student_plaus_scores)*100
    print(f"\n{'='*50}")
    print(f"🎯 BİLGİ DAMITMA BİLANÇOSU (Binary)")
    print(f"   Usta (BERT) Plausibility:     %{bp:.2f}")
    print(f"   Çırak (DistilBERT) Plausibility: %{sp:.2f}")
    print(f"{'='*50}")

## HÜCRE 16: SHAP ANALİZİ — DistilBERT

In [ ]:
import shap
from transformers import pipeline

shap.initjs()

id2label = {0: "Toxic", 1: "Normal"}
student_model.config.id2label = id2label
student_model.config.label2id = {v: k for k, v in id2label.items()}

device = 0 if torch.cuda.is_available() else -1
student_pipe = pipeline("text-classification", model=student_model,
                        tokenizer=student_tokenizer_fast, device=device, top_k=None)

shap_explainer = shap.Explainer(student_pipe)
sample_text = test_tok[42]['text']
true_lbl = test_tok[42]['labels']

print(f"Metin: {sample_text}")
print(f"Gerçek Sınıf: {id2label[true_lbl]}")

shap_values = shap_explainer([sample_text])
shap.plots.text(shap_values)

# Waterfall — Toxic sınıfı için
shap.plots.waterfall(shap_values[0, :, 0], max_display=10)

## HÜCRE 17: OPTİMİZE EDİLMİŞ FİNAL EĞİTİMİ

In [ ]:
from transformers import EarlyStoppingCallback

print("🔥 Optimize Binary DistilBERT Final Eğitimi Başlıyor!")
final_model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

final_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/HateSpeech/distilbert-binary-FINAL",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.05,
    warmup_steps=300,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
)

final_trainer = Trainer(
    model=final_model, args=final_args,
    train_dataset=combined_train, eval_dataset=val_tok,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
final_trainer.train()

## HÜCRE 18: FİNAL TEST SONUÇLARI

In [ ]:
test_tok.set_format(type="torch", columns=cols)
final_preds = final_trainer.predict(test_tok)
final_pred_labels = np.argmax(final_preds.predictions, axis=1)
final_true = [x["labels"].item() for x in test_tok]

print("\n" + "="*60)
print("📋 FİNAL — Optimize Binary DistilBERT Test Sonuçları")
print("="*60)
print(classification_report(final_true, final_pred_labels, target_names=["Toxic","Normal"], digits=4))

final_acc = accuracy_score(final_true, final_pred_labels)
final_f1  = f1_score(final_true, final_pred_labels, average='macro')

print(f"🎯 Accuracy: %{final_acc*100:.2f}")
print(f"🎯 Macro F1:  %{final_f1*100:.2f}")
print(f"\n📊 KARŞILAŞTIRMA:")
print(f"   Eski 3-Sınıf: ~%66 Accuracy, ~%66 F1")
print(f"   Yeni Binary:  %{final_acc*100:.2f} Accuracy, %{final_f1*100:.2f} F1")

## HÜCRE 18.5: FİNAL CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(final_true, final_pred_labels)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Toxic", "Normal"], yticklabels=["Toxic", "Normal"])
plt.title("Optimize DistilBERT Binary - Confusion Matrix")
plt.ylabel("Gerçek Etiket")
plt.xlabel("Tahmin Edilen Etiket")
plt.show()

## HÜCRE 19: MODELİ KAYDET

In [ ]:
final_model.save_pretrained("/content/drive/MyDrive/HateSpeech/binary_model_final")
tokenizer.save_pretrained("/content/drive/MyDrive/HateSpeech/binary_tokenizer_final")
print("✅ Binary model ve tokenizer Drive'a kaydedildi!")